# Обучение нейросети синтеза ДОЭ в Google Colab — глубокая модель (6 стадий уточнения)

**Перед запуском:**
1. *Среда выполнения → Сменить среду выполнения → Аппаратный ускоритель: **GPU A100*** (L4 тоже работает, но примерно вдвое дольше).
2. На Google Диске должна быть папка `MyDrive/neural-doe/` с **новым** файлом `neural_doe_colab.zip`.
3. *Среда выполнения → Выполнить всё*. Вкладку держите открытой.

**Если сессия оборвалась:** заново *Выполнить всё* — обучение продолжится с последнего чекпойнта на Диске (`--resume`), сохранение каждые 500 шагов.

Все результаты (чекпойнты, логи, графики, метрики) пишутся на Диск в `MyDrive/neural-doe/`.

## 1. Видеокарта

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Включите GPU: Среда выполнения → Сменить среду выполнения"

## 2. Google Диск и распаковка проекта

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/neural-doe'
import os
assert os.path.exists(f'{DRIVE}/neural_doe_colab.zip'), f'Нет {DRIVE}/neural_doe_colab.zip — загрузите архив на Диск'
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True); os.makedirs(f'{DRIVE}/results/ml', exist_ok=True)
# всегда распаковываем заново: в живой сессии мог остаться старый код
!rm -rf /content/neural-doe-claude && cp "{DRIVE}/neural_doe_colab.zip" /content/ && cd /content && unzip -q -o neural_doe_colab.zip
%cd /content/neural-doe-claude
!ls

## 3. Зависимости и открытые шрифты (для букв/слов/эмодзи в генераторе)

In [ ]:
FONTS = ('fonts-dejavu fonts-liberation fonts-liberation2 fonts-noto-core fonts-noto-mono fonts-freefont-ttf '
         'fonts-noto-color-emoji fonts-urw-base35 fonts-roboto fonts-open-sans fonts-lato fonts-crosextra-carlito '
         'fonts-crosextra-caladea fonts-ebgaramond fonts-comfortaa fonts-cantarell fonts-font-awesome').split()
!apt-get -qq update > /dev/null
for f in FONTS:  # по одному: отсутствующий пакет не ломает остальные
    !apt-get -qq install -y {f} > /dev/null 2>&1 || echo "нет пакета {f}"
!fc-cache -f > /dev/null
!pip -q install scikit-image pytest


## 4. Проверки: физика/метрики, генератор данных

In [ ]:
!python -m pytest -q tests/test_core.py tests/test_data.py
import os, time, numpy as np
os.environ['DOE_DATA_VERBOSE'] = '1'
from doe.data import QuadGenerator
g = QuadGenerator('train', 0); g.sample(4)
print(g.describe())
assert 'MISSING' not in g.describe(), 'Нет кэша внешних датасетов — проверьте архив'
t = time.time(); b = g.sample(64); print('sample(64):', round(time.time() - t, 3), 's', b.shape, b.dtype)


## 5. Замер скорости
Сравните мс/шаг. На A100: если `batch 32 --amp` медленнее `batch 16` меньше чем в ~1,6 раза, в шаге 6 поставьте `BATCH=32, LR=4e-4, STEPS=20000`.

In [ ]:
INIT = 'checkpoints/init/best.pt'   # = amp_xtalk (модель со штрафом за помехи), к ней добавляем 4 стадии
!python -m doe.train --run bench --init {INIT} --add-stages 4 --bench 20 --batch 16 --xtalk 1.0
!python -m doe.train --run bench --init {INIT} --add-stages 4 --bench 20 --batch 32 --xtalk 1.0 --amp

## 6. Обучение глубокой модели
~16 000 шагов: на A100 ориентировочно ~1–1,5 ч, на L4 ~2,5–3 ч (точное время — в выводе шага 5). В строках `VAL`: `pcc` — КК, `sel` — селективность по глубине, `own` — во скольких плоскостях из 4 своя картинка лучшая. Ориентиры текущей модели amp_xtalk на val: pcc ≈ 0,49, sel ≈ 0,20, own ≈ 3,0.


In [ ]:
# Глубокая модель: amp_xtalk + 4 новые стадии уточнения (всего 6). Новые стадии стартуют как обычные шаги GS
# и учатся их улучшать. Лучший чекпойнт — по Jsel (качество КК/ИСС + селективность по глубине).
RUN, STEPS, BATCH, LR, AMP = 'amp_deep6', 16000, 16, 3e-4, ''   # если на A100 batch 32 + --amp быстрее на картинку: BATCH=32, STEPS=10000, AMP='--amp'
CK, RS = f'{DRIVE}/checkpoints', f'{DRIVE}/results/ml'
!python -m doe.train --run {RUN} --init {INIT} --add-stages 4 --resume \
  --steps {STEPS} --batch {BATCH} --lr {LR} {AMP} --warmup 500 --val-every 1000 --val-n 128 --save-every 500 \
  --qat-frac 0.1 --w-pcc 1.0 --alpha 0.5 --beta 0.25 --gamma 0.3 --norm soft --soft-beta 20 --xtalk 1.0 \
  --best-metric Jsel --ckpt-dir "{CK}" --results-dir "{RS}"

## 7. Итоговая оценка: сеть и сеть + уточнение (20/50/100 шагов), val и test

In [ ]:
BEST = f'{CK}/{RUN}/best.pt'
!PYTHONPATH=. python results/ml/eval_ckpt.py --ckpt "{BEST}" --split val --n 256 --baseline --tag {RUN}
!PYTHONPATH=. python results/ml/eval_ckpt.py --ckpt "{BEST}" --split test --n 256 --baseline --tag {RUN}
!cp results/ml/eval_*{RUN}*.json "{RS}/" 2>/dev/null; ls -la "{RS}"

## Готово
Скачайте с Диска `neural-doe/checkpoints/amp_deep6/best.pt` (именно из папки **amp_deep6**) и папку `neural-doe/results/ml/` — дальше координатор всё проверит независимой оценкой.
